# ROI correlations from the Yeo2011 master table

Load the existing regional master CSV and run the original **mean-feature**
Spearman analysis. This notebook needs no state from `mni_yeo2011_7network.ipynb`:
its required correlation helpers are defined below. It does not reopen clinical
workbooks, re-extract NIfTI statistics, or generate plots.

The master has one row per series/configuration/ROI. The adapter uses `roi_org`
as the canonical ROI identifier and pivots the `mean` column into temporary
`<field_name>_<roi_org>` columns. Each correlation then has one observation per
`(person_id, series_num)`, rather than one observation per ROI row.

With `include_network_means=True`, the analysis also uses the existing seven
bilateral network features: equal averages of regional field means across LH and
RH, calculated **before** correlation. Set it to `False` for regional ROIs only.
No row dropping, covariate adjustment, or multiple-testing correction is added.

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

# Select the source table and the original mean-feature analysis.
field_name = "magnE_mean"
master_path = Path(
    "/projects/p32903/Alex2/ect-workflow/05_analysis/networks/data"
) / f"yeo2011_{field_name}_series_ROI_master.csv"
outcome = "cgi_change"
include_network_means = True

scope = "ROImean_and_network" if include_network_means else "ROImean_only"
output_path = master_path.parent / f"spearman_{field_name}_{scope}_annotated.csv"

## Functions

The statistical helpers match the original notebook. The adapters validate identifiers, duplicate ROI rows, observed ROI coverage, and consistent outcomes before pivoting.

In [ ]:
def load_roi_master(filename: str | Path) -> pd.DataFrame:
    """Read the existing master without filtering or altering its rows."""
    path = Path(filename)
    if not path.is_file():
        raise FileNotFoundError(f"Master CSV does not exist: {path}")
    master = pd.read_csv(path)
    if master.empty:
        raise ValueError(f"Master CSV contains no observations: {path}")
    return master


def summarize_spearman_results(results: pd.DataFrame) -> pd.DataFrame:
    """Report observation counts and regional/bilateral tests by configuration."""
    annotated = results.assign(
        is_regional=results["hemi"].isin(["LH", "RH"]),
        is_bilateral=results["hemi"].eq("both"),
    )
    return annotated.groupby(["field_name", "metric", "model_type", "step"], sort=False).agg(
        n_series=("n", "first"),
        regional_roi_tests=("is_regional", "sum"),
        bilateral_network_tests=("is_bilateral", "sum"),
        total_tests=("x", "size"),
    )


def add_network_means_both_hemis(df: pd.DataFrame, field_name: str = "magnE_mean") -> pd.DataFrame:
    networks = [
        "Vis", "SomMot", "DorsAttn", "SalVentAttn",
        "Limbic", "Cont", "Default",
    ]

    result = df.copy()

    for network in networks:
        columns = []

        for hemisphere in ("LH", "RH"):
            prefix = f"{field_name}_7Networks_{hemisphere}_{network}"

            region_columns = [
                col for col in df.columns
                if col == prefix or col.startswith(prefix + "_")
            ]

            if not region_columns:
                raise ValueError(
                    f"No regional mean columns for {hemisphere} {network}."
                )

            columns.extend(region_columns)

        result[f"{field_name}_{network}_both_hemis"] = df[columns].mean(
            axis=1,
            skipna=False,
        )

    return result


def run_spearman(df: pd.DataFrame, y: str, x: list[str] | str | None = None,) -> pd.DataFrame:
    if df.columns.has_duplicates:
        raise ValueError("Dataframe contains duplicate column names.")

    if y not in df.columns:
        raise KeyError(f"Outcome column {y!r} is missing.")

    if x is None:
        features = [col for col in df.columns if col != y]
    elif isinstance(x, str):
        features = [x]
    else:
        features = list(x)

    if not features:
        raise ValueError("No feature columns selected.")

    if y in features or len(features) != len(set(features)):
        raise ValueError("Features must be unique and must not include y.")

    data = df[[y, *features]].apply(pd.to_numeric, errors="raise")

    if len(data) < 3:
        raise ValueError("At least three observations are required.")

    invalid = data.columns[
        ~np.isfinite(data.to_numpy(dtype=float)).all(axis=0)
    ].tolist()
    if invalid:
        raise ValueError(f"Missing or non-finite values in: {invalid}")

    constant = data.columns[data.nunique().lt(2)].tolist()
    if constant:
        raise ValueError(f"Constant columns cannot be correlated: {constant}")

    results = []
    for feature in features:
        rho, p_value = spearmanr(
            data[feature],
            data[y],
            nan_policy="raise",
        )
        results.append({
            "x": feature,
            "y": y,
            "n": len(data),
            "rho": float(rho),
            "p_value": float(p_value),
        })

    return pd.DataFrame(results)


def annotate_correlation_features(results: pd.DataFrame) -> pd.DataFrame:
    """Parse feature labels using the exact source field, never underscore guesses."""
    required = {"x", "field_name", "metric"}
    missing = required - set(results.columns)
    if missing:
        raise KeyError(f"Missing feature identity columns: {sorted(missing)}")

    result = results.copy()
    parents = "Vis|SomMot|DorsAttn|SalVentAttn|Limbic|Cont|Default"
    metadata = []

    for feature, source_field, metric_name in result[["x", "field_name", "metric"]].itertuples(index=False, name=None):
        if not isinstance(source_field, str) or not source_field or source_field != source_field.strip():
            raise ValueError(f"Invalid source field_name: {source_field!r}")
        if not isinstance(metric_name, str) or not metric_name:
            raise ValueError(f"Invalid ROI summary metric: {metric_name!r}")

        prefix = f"{source_field}_"
        if not isinstance(feature, str) or not feature.startswith(prefix):
            raise ValueError(f"Feature {feature!r} does not preserve field_name={source_field!r}.")

        label = feature[len(prefix):]
        regional = re.fullmatch(rf"7Networks_(LH|RH)_({parents})(?:_(.+))?", label)
        bilateral = re.fullmatch(rf"({parents})_both_hemis", label)

        if regional:
            hemi, network, component = regional.groups()
            metadata.append((label, component or network, hemi, network))
        elif bilateral:
            network = bilateral.group(1)
            metadata.append((None, network, "both", network))
        else:
            raise ValueError(f"Unrecognized ROI/network label in feature {feature!r}.")

    parsed = pd.DataFrame(metadata, columns=["roi_org", "roi", "hemi", "network"], index=result.index)
    for column in parsed:
        if column in result.columns:
            agrees = result[column].fillna("").eq(parsed[column].fillna(""))
            if not agrees.all():
                raise ValueError(f"Existing {column!r} metadata conflicts with x and field_name.")
        result[column] = parsed[column]

    return result


def save_spearman_table(df: pd.DataFrame, filename) -> pd.DataFrame:
    """Save source-aware feature names and explicit field/metric/ROI metadata."""
    columns = ["x", "y", "n", "rho", "p_value", "field_name", "metric", "model_type", "step"]
    missing = set(columns) - set(df.columns)
    if missing:
        raise KeyError(f"Missing correlation columns: {sorted(missing)}")
    result = annotate_correlation_features(df[columns])
    order = [*columns, "roi_org", "roi", "hemi", "network"]
    result = result[order]
    output_path = Path(filename)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    result.to_csv(output_path, index=False)
    return result


def roi_master_to_mean_tables(master: pd.DataFrame, field_name: str = "magnE_mean", y: str = "cgi_change") -> dict:
    """Convert ROI rows into the notebook's wide mean-feature analysis tables."""
    series_keys = ["person_id", "series_num"]
    config_keys = ["model_type", "step"]

    required = [
        *series_keys, *config_keys,
        "field_name", "roi_org", "mean", y,
    ]

    if master.columns.has_duplicates:
        raise ValueError("Master contains duplicate column names.")

    missing = set(required) - set(master.columns)
    if missing:
        raise KeyError(f"Missing master columns: {sorted(missing)}")

    selected = master.loc[
        master["field_name"].eq(field_name)
    ].copy()

    if selected.empty:
        raise ValueError(f"No master rows for field_name={field_name!r}.")

    identity = [*series_keys, *config_keys, "roi_org"]

    if selected[identity].isna().any().any():
        raise ValueError("Missing series, configuration, or ROI identifiers.")

    if selected.duplicated(identity).any():
        raise ValueError("Duplicate series/configuration/ROI rows.")

    # CGI must agree across every ROI/configuration belonging to a series.
    outcome_counts = selected.groupby(series_keys)[y].nunique(dropna=False)

    if outcome_counts.ne(1).any():
        raise ValueError(
            f"Inconsistent {y!r} values within treatment series."
        )

    roi_names = selected["roi_org"].drop_duplicates().tolist()
    tables = {}

    for configuration, group in selected.groupby(config_keys, sort=False):
        roi_counts = group.groupby(series_keys)["roi_org"].nunique()

        if roi_counts.ne(len(roi_names)).any():
            raise ValueError(
                f"{configuration}: incomplete ROI coverage for a series."
            )

        # Pivot directly: duplicate observations are never averaged.
        wide = group.pivot(
            index=series_keys,
            columns="roi_org",
            values="mean",
        ).reindex(columns=roi_names)

        # Repeated outcome values were checked above before deduplication.
        outcome = (
            group[series_keys + [y]]
            .drop_duplicates(series_keys)
            .set_index(series_keys)[y]
        )

        wide = wide.rename(columns=lambda roi: f"{field_name}_{roi}")
        wide.columns.name = None

        # Align outcomes by person/series keys, not row position.
        wide.insert(0, y, outcome.reindex(wide.index))

        tables[configuration] = wide

    return tables


def run_master_spearman(master: pd.DataFrame, field_name: str = "magnE_mean", y: str = "cgi_change", include_network_means: bool = True) -> pd.DataFrame:
    """Run the notebook's original mean-feature analysis from the new master."""
    tables = roi_master_to_mean_tables(
        master,
        field_name=field_name,
        y=y,
    )

    results = []

    for (model_type, step), table in tables.items():
        if include_network_means:
            table = add_network_means_both_hemis(table, field_name=field_name)

        features = [
            column for column in table.columns
            if column != y
        ]

        correlations = run_spearman(
            table,
            y=y,
            x=features,
        )

        results.append(
            correlations.assign(
                model_type=model_type,
                step=step,
                field_name=field_name,
                metric="mean",
            )
        )

    return pd.concat(results, ignore_index=True)

## Load the master table

In [ ]:
df_master = load_roi_master(master_path)
print(f"Loaded {len(df_master):,} ROI rows and {len(df_master.columns)} columns from {master_path}")

## Run the correlations

Filter the selected field and run each available model/step configuration separately. Invalid numeric values, constant columns, and incomplete ROI observations raise errors.

In [ ]:
df_spearman_all = run_master_spearman(
    df_master,
    field_name=field_name,
    y=outcome,
    include_network_means=include_network_means,
)
print(f"Computed {len(df_spearman_all)} correlations for {field_name}, metric=mean, outcome={outcome}.")

## Save the annotated correlation table

Saving is a separate step. Change `output_path` in the settings cell to choose another destination.

In [ ]:
df_spearman_annotated = save_spearman_table(df_spearman_all, output_path)

# This dataframe also has the feature names expected by the existing plot helpers.
all_results = df_spearman_annotated
print(f"Saved annotated correlations to {output_path}")

## Check the results

For the supplied complete four-configuration master, each configuration has
35 treatment series and 51 regional ROIs. Expect 51 tests per configuration
with regional features only, or 58 with the seven bilateral means included:
**204 or 232 total tests**, all with **n = 35**.

The export schema is:
`x, y, n, rho, p_value, field_name, metric, model_type, step, roi_org, roi, hemi, network`.
Each `x` preserves the exact source field: for example,
`magnE_mean_7Networks_LH_DorsAttn_FEF`, `E_7Networks_LH_DorsAttn_FEF`,
or `magnJ_7Networks_LH_DorsAttn_FEF`. The separate `metric` column is
`mean` for this analysis. Field names are matched as literal prefixes rather
than inferred by splitting underscores. `roi_org` is the complete atlas label
for regional rows and blank for bilateral network aggregates. The saved master table is unchanged.
P-values are unadjusted, and `n` counts series, not distinct persons.

In [ ]:
summarize_spearman_results(df_spearman_annotated)

In [ ]:
df_spearman_annotated.head(10)